In [ ]:
base_url = 'http://www.carsurvey.org'

In [ ]:
import requests
from bs4 import BeautifulSoup

def extract_most_popular_brands(base_url = base_url):
  response = requests.get(base_url)
  soup = BeautifulSoup(response.text, 'html.parser')
  most_popular_section = soup.find('h2', class_='most-popular-header')
  manufacturer_list = most_popular_section.find_next('ol', class_='manufacturer-list')
  popular_makes = []
  for li in manufacturer_list.find_all('li'):
      a = li.find('a')
      if a:
          popular_makes.append({
              'name': a.text.strip(),
              'url': f"{base_url}{a['href'].strip()}"
          })
  return popular_makes

most_popular_brands = extract_most_popular_brands(base_url)
most_popular_brands

[{'name': 'BMW', 'url': 'http://www.carsurvey.org/reviews/bmw/'},
 {'name': 'Buick', 'url': 'http://www.carsurvey.org/reviews/buick/'},
 {'name': 'Chevrolet', 'url': 'http://www.carsurvey.org/reviews/chevrolet/'},
 {'name': 'Chrysler', 'url': 'http://www.carsurvey.org/reviews/chrysler/'},
 {'name': 'Citroen', 'url': 'http://www.carsurvey.org/reviews/citroen/'},
 {'name': 'Dodge', 'url': 'http://www.carsurvey.org/reviews/dodge/'},
 {'name': 'Fiat', 'url': 'http://www.carsurvey.org/reviews/fiat/'},
 {'name': 'Ford', 'url': 'http://www.carsurvey.org/reviews/ford/'},
 {'name': 'Honda', 'url': 'http://www.carsurvey.org/reviews/honda/'},
 {'name': 'Hyundai', 'url': 'http://www.carsurvey.org/reviews/hyundai/'},
 {'name': 'Jeep', 'url': 'http://www.carsurvey.org/reviews/jeep/'},
 {'name': 'Kia', 'url': 'http://www.carsurvey.org/reviews/kia/'},
 {'name': 'Mazda', 'url': 'http://www.carsurvey.org/reviews/mazda/'},
 {'name': 'Mercedes-Benz',
  'url': 'http://www.carsurvey.org/reviews/mercedes-ben

In [ ]:
def extract_models(url, base_url=base_url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    model_table = soup.find('table', class_='model-list')
    if not model_table:
        raise ValueError("Couldn't find the table with class 'model-list'")

    models = []
    for tr in model_table.find_all('tr'):
        a = tr.find('a')
        if a:
            models.append(base_url+a['href'].strip())
    return models

extract_models('http://www.carsurvey.org/reviews/bmw/')

['http://www.carsurvey.org/reviews/bmw/1_series/',
 'http://www.carsurvey.org/reviews/bmw/2_series/single-page/',
 'http://www.carsurvey.org/reviews/bmw/2002/single-page/',
 'http://www.carsurvey.org/reviews/bmw/3_series/',
 'http://www.carsurvey.org/reviews/bmw/4_series/',
 'http://www.carsurvey.org/reviews/bmw/5_series/',
 'http://www.carsurvey.org/reviews/bmw/507/',
 'http://www.carsurvey.org/reviews/bmw/6_series/',
 'http://www.carsurvey.org/reviews/bmw/7_series/',
 'http://www.carsurvey.org/reviews/bmw/8_series/',
 'http://www.carsurvey.org/reviews/bmw/cs_coupe/',
 'http://www.carsurvey.org/reviews/bmw/e3/',
 'http://www.carsurvey.org/reviews/bmw/i3/',
 'http://www.carsurvey.org/reviews/bmw/i8/',
 'http://www.carsurvey.org/reviews/bmw/m_coupe/single-page/',
 'http://www.carsurvey.org/reviews/bmw/m_roadster/single-page/',
 'http://www.carsurvey.org/reviews/bmw/m3/',
 'http://www.carsurvey.org/reviews/bmw/m5/',
 'http://www.carsurvey.org/reviews/bmw/m6/',
 'http://www.carsurvey.org/

In [ ]:
def split_by_years(url, base_utl=base_url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    model_year_table = soup.find('table', class_='model-year-list')
    if not model_year_table:
        return [url]

    models = []
    for tr in model_year_table.find_all('tr'):
        a = tr.find('a')
        if a:
            models.append(base_url + a['href'].strip())
    return models

In [ ]:
split_by_years('http://www.carsurvey.org/reviews/bmw/2_series/single-page/')

['http://www.carsurvey.org/reviews/bmw/2_series/single-page/']

In [ ]:
def get_review_link(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    model_table = soup.find('table', class_='review-list-table')
    if not model_table:
        raise ValueError("model-list finding issue")

    models = []
    for tr in model_table.find_all('tr'):
        a = tr.find('a')
        if a and a.get('href'):
            models.append(base_url+a['href'].strip())
    return models

get_review_link('http://www.carsurvey.org/reviews/bmw/2_series/single-page/')

['http://www.carsurvey.org/reviews/bmw/2_series/#r144667',
 'http://www.carsurvey.org/reviews/bmw/2_series/page-2/',
 'http://www.carsurvey.org/reviews/bmw/2_series/#r146256',
 'http://www.carsurvey.org/reviews/bmw/2_series/']

In [ ]:
from tqdm import tqdm
def get_all_reviews_for_brand(brand_info, base_url = base_url):
  res = []
  print(f"  Extract info for {brand_info['name']}")
  brand_link = brand_info['url']
  models = extract_models(brand_link)
  for model in models:
    models_by_years = split_by_years(model)
    for model_year in models_by_years:
      try:
        res += get_review_link(model_year)
      except:
        res.append(model_year)

  return res

def get_all_reviews(brand_links):
  res = []
  for brand_link in tqdm(brand_links):
    res += get_all_reviews_for_brand(brand_link)
  return res


reviews_link = get_all_reviews(most_popular_brands[:2])

  0%|          | 0/2 [00:00<?, ?it/s]

  Extract info for BMW


 50%|█████     | 1/2 [01:25<01:25, 85.62s/it]

  Extract info for Buick


100%|██████████| 2/2 [02:54<00:00, 87.18s/it]


In [ ]:
import json

def extract_json(link):
  res = {}
  response = requests.get(link)
  soup = BeautifulSoup(response.text, 'html.parser')
  article = soup.find('article', class_='cf single-review')
  if not article:
    return
  for topic in article.find_all('section'):
    topic_name = topic.find('h2')
    if not topic_name:
      continue
    topic_name = topic_name.get_text(strip=True)
    topic_text = topic.find_all('p')
    text = "".join([p.get_text(strip=True) for p in topic_text])
    res[topic_name] = text

  table = article.find('table')
  rows = table.find_all('tr')
  for row in rows:
      cells = row.find_all('td')
      if len(cells) == 2:
          key = cells[0].get_text(strip=True)
          value = cells[1].get_text(strip=True)
          res[key] = value
  return json.dumps(res, indent=2)

print(extract_json('http://www.carsurvey.org/reviews/bmw/1_series/2017/'))

{
  "Summary:": "Performance bargain with real character",
  "Faults:": "Brake squeal in reverse (cured itself after about 5,000 miles).No other faults.",
  "General Comments:": "It's exceeded every expectation.The B58 turbo six pot is the real star of the show, and I've never known another engine that seems to adapt to your mood or your driving style so effectively. Cruising about, it's silky smooth and pretty quiet. In Sport+ with the exhaust flaps open, it howls and growls like all the best BMW sixes, and the performance it delivers is fantastic. Any gear, any revs, it just takes off like a scalded cat. It hasn't used a drop of oil in 29,000 miles, or so much as misfired, and is returning, with a bit of motorway use admittedly, a staggering 30 MPG average. People whine that it doesn't have the character of the old NA BMW engines, and maybe it doesn't, but it's in a different league to the four pot units in the competition. To get anything that even competes with it, you're looking a

In [ ]:
json_reviews = [extract_json(link) for link in tqdm(reviews_link)]

100%|██████████| 3446/3446 [21:20<00:00,  2.69it/s]
